# Kronecker Solve Rewrite From Scratch

This notebook is a hands-on implementation of a PyTensor graph rewrite for:

- naive pattern: `solve(kron(A, B), x)`
- rewritten pattern: factorized solve using two smaller linear solves

The goal is to move from theory to concrete graph manipulation (`graph_replace`, `node_rewriter`, and `rewrite_graph`).

## 0. Imports

In [20]:
import time

import numpy as np
import pytensor
import pytensor.tensor as pt

from pytensor import graph_replace
from pytensor.graph.rewriting.basic import in2out, node_rewriter
from pytensor.graph.rewriting.utils import rewrite_graph
from pytensor.tensor.blockwise import Blockwise
from pytensor.tensor.linalg import kron, solve
from pytensor.tensor.nlinalg import KroneckerProduct
from pytensor.tensor.slinalg import Solve

np.random.seed(123)
print(f"PyTensor version: {pytensor.__version__}")

PyTensor version: 2.38.2


## 1. Quick Check of the Wrong Formula Mentioned on [Discourse](https://discourse.pymc.io/t/gsoc-2026-interest-linear-algebra-rewrites/17580/5#:~:text=Now%20that%20you%20see%20the%20two%20ways%20to%20compute%20a%20kronecker%2C%20can%20you%20use%20graph_replace%20to%20swap%20out%20a%20graph%20with%20y%20%3D%20solve(kron(a%2C%20b)%2C%20x)%20with%20y%20%3D%20kron(solve(a%2C%20x%5B%3Aa.shape%5B1%5D%5D)%2C%20solve(b%2C%20x%5Ba.shape%3A%5D))%20%3F)

Before the full rewrite, we test the literal expression as written in the discussion:

`y = kron(solve(A, x[:A.shape[1]]), solve(B, x[A.shape[1]:]))`

This may seems a useful directional idea, but it is not shape-correct as a general replacement for `solve(kron(A, B), x)`. We test a finite-slice interpretation and compare it to the true solution.

In [ ]:
# size
n1_lit, n2_lit = 5, 4

# PyTensor random data generation
A_draw = pt.random.normal(size=(n1_lit, n1_lit)) + 2.0 * pt.eye(n1_lit)
B_draw = pt.random.normal(size=(n2_lit, n2_lit)) + 2.0 * pt.eye(n2_lit)
x_draw = pt.random.normal(size=(n1_lit * n2_lit,))
f_draw = pytensor.function([], [A_draw, B_draw, x_draw])
A_lit, B_lit, x_lit = f_draw()

# PyTensor implementation of the full naive solution solve(kron(A, B), x)
A_lit_sym = pt.matrix("A_lit")
B_lit_sym = pt.matrix("B_lit")
x_lit_sym = pt.vector("x_lit")

full_naive_expr = solve(kron(A_lit_sym, B_lit_sym), x_lit_sym)
f_full_naive = pytensor.function([A_lit_sym, B_lit_sym, x_lit_sym], full_naive_expr)
y_true_lit = f_full_naive(A_lit, B_lit, x_lit)

# PyTensor implementation of the literal finite-slice interpretation
u_lit_expr = solve(A_lit_sym, x_lit_sym[:n1_lit])
v_lit_expr = solve(
    B_lit_sym, x_lit_sym[n1_lit : n1_lit + n2_lit]
)  # here i am forced to use a limited slice to match dimensions
y_literal_expr = pt.outer(u_lit_expr, v_lit_expr).ravel()
f_literal = pytensor.function([A_lit_sym, B_lit_sym, x_lit_sym], y_literal_expr)
y_literal_lit = f_literal(A_lit, B_lit, x_lit)

# error analysis
literal_err_lit = np.max(np.abs(y_true_lit - y_literal_lit))
used_fraction_lit = (n1_lit + n2_lit) / (n1_lit * n2_lit)

print(f"Literal sliced form max error: {literal_err_lit:.3e}")
print(f"Fraction of x entries used: {used_fraction_lit:.2%}")
print("We therefore use the reshape-based factorized solve in the rest of the notebook.")

Literal sliced form max error: 1.403e+00
Fraction of x entries used: 45.00%
We therefore use the reshape-based factorized solve in the rest of the notebook.


## 2. Build a Naive Symbolic Graph

We start now the correct rewrite development step by step.

In [22]:
A_sym = pt.matrix("A")
B_sym = pt.matrix("B")
x_sym = pt.vector("x")

naive_expr = solve(kron(A_sym, B_sym), x_sym)

print("Naive graph:")
pytensor.dprint(naive_expr, depth=3)

Naive graph:
Blockwise{Solve{assume_a='gen', lower=False, b_ndim=1, overwrite_a=False, overwrite_b=False}, (m,m),(m)->(m)} [id A]
 ├─ KroneckerProduct{inline=False} [id B]
 │  ├─ A [id C]
 │  └─ B [id D]
 └─ x [id E]

Inner graphs:

KroneckerProduct{inline=False} [id B]
 ← Reshape{2} [id F]
    ├─ Mul [id G]
    │  ├─ ExpandDims{axes=[1, 3]} [id H]
    │  └─ ExpandDims{axes=[0, 2]} [id I]
    └─ MakeVector{dtype='int64'} [id J]
       ├─ Mul [id K]
       └─ Mul [id L]


## 3. Manual Replacement With `graph_replace`

For row-major flattening (`reshape`/`ravel` default behavior), a correct factorized replacement is:

$$
y = \operatorname{solve}(A \otimes B, x)
\quad\Longleftrightarrow\quad
Y = \operatorname{solve}(A, \operatorname{solve}(B, X^T)^T),
$$

where $X = \operatorname{reshape}(x, (n_A, n_B))$ and $y = \operatorname{ravel}(Y)$.

In [23]:
nA = A_sym.shape[-1]
nB = B_sym.shape[-1]

X_mat = x_sym.reshape((nA, nB))
rewritten_expr_manual = solve(A_sym, solve(B_sym, X_mat.T).T).ravel()

rewritten_graph_manual = graph_replace(naive_expr, {naive_expr: rewritten_expr_manual})

print("Manual rewritten graph:")
pytensor.dprint(rewritten_graph_manual, depth=4)

Manual rewritten graph:
Reshape{1} [id A]
 ├─ Blockwise{Solve{assume_a='gen', lower=False, b_ndim=2, overwrite_a=False, overwrite_b=False}, (m,m),(m,n)->(m,n)} [id B]
 │  ├─ A [id C]
 │  └─ Transpose{axes=[1, 0]} [id D]
 │     └─ Blockwise{Solve{assume_a='gen', lower=False, b_ndim=2, overwrite_a=False, overwrite_b=False}, (m,m),(m,n)->(m,n)} [id E]
 └─ [-1] [id F]


## 4. Numerical Correctness Check

We compare the naive and rewritten expressions on random invertible matrices (not necessarily symmetric).

In [ ]:
def random_invertible(n, seed_offset=0):
    rs = pt.random.RandomStream(seed=2026 + seed_offset)
    M = rs.normal(size=(n, n)) + 2.0 * pt.eye(n)  # ensure invertibility and good conditioning
    draw_fn = pytensor.function([], M)
    return draw_fn().astype(np.float64)


n1, n2 = 5, 4
A_np = random_invertible(n1, 0)
B_np = random_invertible(n2, 1)

x_rs = pt.random.RandomStream(seed=2028)
x_draw = x_rs.normal(size=(n1 * n2,))
x_np = pytensor.function([], x_draw)().astype(np.float64)

f_naive = pytensor.function([A_sym, B_sym, x_sym], naive_expr)
f_manual = pytensor.function([A_sym, B_sym, x_sym], rewritten_graph_manual)

y_naive = f_naive(A_np, B_np, x_np)
y_manual = f_manual(A_np, B_np, x_np)

y1_sym = pt.vector("y1")
y2_sym = pt.vector("y2")
err_expr = pt.max(pt.abs(y1_sym - y2_sym))
f_max_err = pytensor.function([y1_sym, y2_sym], err_expr)
max_err = f_max_err(y_naive, y_manual)

print(f"max |naive - manual| = {max_err:.3e}")

max |naive - manual| = 6.661e-16


## 5. Automatic Pattern Detection With `node_rewriter`

Now we automate the replacement for matching nodes in a graph. This is the core building block toward production rewrites.

In [25]:
@node_rewriter(tracks=[Blockwise])
def local_solve_kron_vector(fgraph, node):
    # We target Blockwise(Solve) nodes
    if not (isinstance(node.op, Blockwise) and isinstance(node.op.core_op, Solve)):
        return None

    A_kron, rhs = node.inputs

    # Match kron(A, B) exactly (2-factor case for this prototype)
    kron_node = A_kron.owner
    if kron_node is None or not isinstance(kron_node.op, KroneckerProduct):
        return None
    if len(kron_node.inputs) != 2:
        return None

    A, B = kron_node.inputs

    # Restrict to vector rhs for a minimal, safe first rewrite
    if rhs.type.ndim != 1:
        return None

    nA = A.shape[-1]
    nB = B.shape[-1]
    rhs_mat = rhs.reshape((nA, nB))

    # Row-major compatible factorized solve
    out = solve(A, solve(B, rhs_mat.T).T).ravel()
    return [out]


auto_rewritten_expr = rewrite_graph(
    naive_expr,
    custom_rewrite=in2out(local_solve_kron_vector, name="local_solve_kron_vector"),
    clone=False,
)

print("Auto rewritten graph:")
pytensor.dprint(auto_rewritten_expr, depth=4)

f_auto = pytensor.function([A_sym, B_sym, x_sym], auto_rewritten_expr)
y_auto = f_auto(A_np, B_np, x_np)
auto_err = f_max_err(y_naive, y_auto)
print(f"max |naive - auto| = {auto_err:.3e}")

Auto rewritten graph:
Reshape{1} [id A]
 ├─ Blockwise{Solve{assume_a='gen', lower=False, b_ndim=2, overwrite_a=False, overwrite_b=False}, (m,m),(m,n)->(m,n)} [id B]
 │  ├─ A [id C]
 │  └─ Transpose{axes=[1, 0]} [id D]
 │     └─ Blockwise{Solve{assume_a='gen', lower=False, b_ndim=2, overwrite_a=False, overwrite_b=False}, (m,m),(m,n)->(m,n)} [id E]
 └─ [-1] [id F]
max |naive - auto| = 6.661e-16


## 6. Benchmark

This is just a sanity benchmark for the prototype expression-level rewrite.

In [26]:
n1, n2 = 22, 18
A_np = random_invertible(n1, 11)
B_np = random_invertible(n2, 12)

x_rs_bench = pt.random.RandomStream(seed=2050)
x_draw_bench = x_rs_bench.normal(size=(n1 * n2,))
x_np = pytensor.function([], x_draw_bench)().astype(np.float64)

_ = f_naive(A_np, B_np, x_np)
_ = f_auto(A_np, B_np, x_np)

N = 50
t0 = time.perf_counter()
for _ in range(N):
    y_n = f_naive(A_np, B_np, x_np)
t_naive = (time.perf_counter() - t0) / N

t0 = time.perf_counter()
for _ in range(N):
    y_a = f_auto(A_np, B_np, x_np)
t_auto = (time.perf_counter() - t0) / N

bench_err = f_max_err(y_n, y_a)
print(f"naive: {t_naive * 1e3:.3f} ms/eval")
print(f"auto : {t_auto * 1e3:.3f} ms/eval")
print(f"speedup: {t_naive / t_auto:.2f}x")
print(f"max error: {bench_err:.3e}")

naive: 14.442 ms/eval
auto : 0.290 ms/eval
speedup: 49.76x
max error: 8.044e-14
